In [8]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

In [9]:
uav_path = 'Wegrzce_tify/01W1509.tif'

UAV_BANDS = {
    'blue_450': 3,   # 442.7 nm
    'green_531': 7,  # 522.1 nm
    'green_550': 8,  # 542.9 nm
    'green_570': 9,  # 561.5 nm
    'red_680': 15,   # 682.4 nm
    're_700': 16,    # 702.7 nm
    're_705': 16,    # 702.7 nm
    'nir_750': 18,   # 741.2 nm
    'nir_800': 21    # 801.8 nm
}

In [ ]:
d = {}
with rasterio.open(uav_path) as src:
    for name, b_idx in UAV_BANDS.items():
        band = src.read(b_idx).astype('float32')
        if src.nodata is not None:
            band[band == src.nodata] = np.nan
        d[name] = band

# ==========================================
# OBLICZANIE WSKAŹNIKÓW 
# ==========================================
eps = 1e-8
indices = {}

# 1. NDVI - 800, 680
indices['NDVI'] = (d['nir_800'] - d['red_680']) / (d['nir_800'] + d['red_680'] + eps)

# 2. EVI - 800, 680, 450
indices['EVI'] = 2.5 * ((d['nir_800'] - d['red_680']) / 
                        (d['nir_800'] + 6 * d['red_680'] - 7.5 * d['blue_450'] + 1))

# 3. SAVI - 800, 680, L=0.50
L = 0.50
indices['SAVI'] = ((d['nir_800'] - d['red_680']) / 
                   (d['nir_800'] + d['red_680'] + L + eps)) * (1 + L)

# 4. RENDVI - 750, 705
indices['RENDVI'] = (d['nir_750'] - d['re_705']) / (d['nir_750'] + d['re_705'] + eps)

# 5. PRI - 531, 570
indices['PRI'] = (d['green_531'] - d['green_570']) / (d['green_531'] + d['green_570'] + eps)

# 6. ARI - 550, 700
indices['ARI'] = (1 / (d['green_550'] + eps)) - (1 / (d['re_700'] + eps))

# 7. CIre (Chlorophyll Index Red-Edge) - 800, 705
indices['CIre'] = (d['nir_800'] / (d['re_705'] + eps)) - 1.0

C:\Users\user\AppData\Local\Temp\ipykernel_8676\4111022936.py:4: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  band = src.read(b_idx).astype('float32')


: 

In [ ]:
# ==========================================
# 3. WIZUALIZACJA 
# ==========================================
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

plots_config = [
    ('NDVI', 'RdYlGn', 0.1, 0.9),
    ('EVI', 'RdYlGn', 0.0, 0.8),
    ('SAVI', 'RdYlGn', 0.1, 0.7),
    ('RENDVI', 'RdYlGn', 0.0, 0.6),
    ('PRI', 'viridis', -0.1, 0.1),
    ('ARI', 'viridis', 0.0, 0.05),
    ('CIre', 'viridis', 0.0, 3.0) 
]

for i, (name, cmap, vmin, vmax) in enumerate(plots_config):
    # Możesz pominąć [::step, ::step], bo macierze są już małe
    im = axes[i].imshow(indices[name], cmap=cmap, vmin=vmin, vmax=vmax)
    axes[i].set_title(name, fontsize=14, fontweight='bold')
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

fig.delaxes(axes[7])

plt.tight_layout()
plt.savefig('wskazniki_z_drona_figspec_7_wykresow.png', dpi=300, bbox_inches='tight')
plt.show()
